# Filtrado de videos y seleccion de escenas
En este notebook se realizara el filtrado de videos en la base de datos del reto y se seleccionaran las escenas significativas para la recaudacción de datos.

## Database and ROI setup

In [ ]:
import psycopg2
import json
import geopandas as gpd
import pandas as pd

# Database Connection Using psycopg2
POSTGIS_CONN = {
    "dbname": "crowdcounting",
    "user": "admin",
    "password": "admin",
    "host": "40.84.231.179",
    "port": "5434",
}

In [17]:
%pip install folium>=0.12 matplotlib mapclassify

roi = gpd.read_file('../playgroundROI.gpkg')
roi.explore()

Note: you may need to restart the kernel to use updated packages.


In [18]:
import geopandas as gpd
import pandas as pd
import psycopg2
from shapely.ops import unary_union

# 1) Carga ROI y asegúrate de EPSG:4326
roi = gpd.read_file('../playgroundROI.gpkg')
if roi.crs is None:
    raise ValueError("El GeoPackage no tiene CRS. Asigna uno correcto o reproyecta.")
if roi.crs.to_epsg() != 4326:
    roi = roi.to_crs(epsg=4326)

# Un solo polígono para la consulta (une si hay varios)
roi_geom = unary_union(roi.geometry)
roi_wkt = roi_geom.wkt  # lo enviaremos como parámetro WKT

# 2) Consulta con sjoin (ST_Intersects) + LIMIT 1000 + conteo total con ventana
SQL_QUERY = """
WITH roi as (
  SELECT ST_GeomFromText(%s, 4326) AS geom
),
filtered as (
  SELECT p.*
  FROM person_observed p
  JOIN roi r
    ON ST_Intersects(p.geom, r.geom)
)
SELECT
  *,
  COUNT(*) OVER() AS total_matches
FROM filtered
ORDER BY timestamp
"""

try:
    conn = psycopg2.connect(**POSTGIS_CONN)
    df = pd.read_sql(SQL_QUERY, conn, params=[roi_wkt])
    conn.close()
except Exception as e:
    print(f"Error connecting to database: {e}")
    df = pd.DataFrame()

# 3) Mostrar resultados y total
if not df.empty:
    total = int(df.loc[0, 'total_matches'])
    print(f"Total dentro del ROI: {total}")
    print(f"Mostrando {len(df)} filas (máximo 1000).")
else:
    print("No hubo coincidencias dentro del ROI o la consulta falló.")

/tmp/ipykernel_2643246/3383462602.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(SQL_QUERY, conn, params=[roi_wkt])


Total dentro del ROI: 1592490
Mostrando 1592490 filas (máximo 1000).


## El filtrado

In [19]:
import pandas as pd

# Asume que `df` ya existe en el kernel (proveniente de reading-db.ipynb)
# Ajusta estos parámetros según necesidad:
MIN_DETECTIONS = 50    # mínimo detecciones para considerar una escena
MAX_SCENES = 100       # cap si quieres top-N

# Derivar video_id y normalizar timestamp
df = df.copy()
df['video_id'] = df['id'].str.split(':').str[0]
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Agrupar por video + cámara y resumir
groups = df.groupby(['video_id', 'camera_name'])
scenes = []
for (vid, cam), g in groups:
    n = len(g)
    if n < MIN_DETECTIONS:
        continue
    t_start = g['timestamp'].min()
    t_end = g['timestamp'].max()
    blob_path = vid  # o construir ruta completa si se requiere
    scenes.append({
        'video_id': vid,
        'camera': cam,
        't_start': t_start.isoformat(),
        't_end': t_end.isoformat(),
        'blob_path': blob_path,
        'n_detections': n
    })

scenes_df = pd.DataFrame(scenes).sort_values('n_detections', ascending=False)
scenes_df = scenes_df.head(MAX_SCENES)
scenes_df.to_csv('data/videos.csv', index=False)
print(f"Saved {len(scenes_df)} scenes to data/videos.csv")

Saved 100 scenes to data/videos.csv


## Descarga para verificacion de actividad de videos

In [ ]:
import os
from datetime import datetime
from azure.storage.blob import BlobServiceClient
from tqdm import tqdm

# CSV relativo al notebook
CSV = "data/videos.csv"  # o path absoluto: /home/sformador/equipo2/Socio-Formador-IA-Avanzada/Playground/data/videos.csv
OUTDIR = "Downloads"
N = 5  # cuántos videos descargar desde el inicio del CSV (ajusta o usa indices)

account_url = 'https://cienciaciudades2024.blob.core.windows.net'
container = 'crowdcounting'
sas_token = 'sp=racwdl&st=2025-02-05T16:48:09Z&se=2026-04-02T00:48:09Z&spr=https&sv=2022-11-02&sr=c&sig=pXOsajenPg9lvILk6OozoyE%2BUz%2FDXivaBWwIioLyRVo%3D'
blob_service_client = BlobServiceClient(account_url, credential=sas_token)
azure_path = df.iloc[-1]['id'].split(':')[0]
if isinstance(azure_path, str):
    azure_path = [azure_path]

container_client = blob_service_client.get_container_client(container)

download_paths = []

for path in azure_path:
    print(f"Downloading {path}...")
    blob_client = container_client.get_blob_client(path)
    download_file_path = 'Downloads/' + path
    os.makedirs(os.path.dirname(download_file_path), exist_ok=True)
    # Download the blob
    with open(download_file_path, "wb") as download_file:
        download_stream = blob_client.download_blob()
        download_file.write(download_stream.readall())
    download_paths.append(download_file_path)

In [20]:
import os
import pandas as pd
from azure.storage.blob import BlobServiceClient

CSV = "/home/sformador/equipo2/Socio-Formador-IA-Avanzada/Playground/data/videos.csv"
OUTDIR = "Downloads"
N = 5  # cuántos videos descargar (cabeza del CSV)

# Copia los valores desde reading-db.ipynb si no están en el kernel
account_url = 'https://cienciaciudades2024.blob.core.windows.net'
container = 'crowdcounting'
sas_token = 'sp=racwdl&st=2025-02-05T16:48:09Z&se=2026-04-02T00:48:09Z&spr=https&sv=2022-11-02&sr=c&sig=pXOsajenPg9lvILk6OozoyE%2BUz%2FDXivaBWwIioLyRVo%3D'

df_csv = pd.read_csv(CSV)
os.makedirs(OUTDIR, exist_ok=True)

client = BlobServiceClient(account_url=account_url, credential=sas_token)
container_client = client.get_container_client(container)

for p in df_csv['blob_path'].head(N):
    print("Downloading", p)
    blob_client = container_client.get_blob_client(p)
    out_path = os.path.join(OUTDIR, os.path.basename(p))
    with open(out_path, "wb") as f:
        stream = blob_client.download_blob()
        f.write(stream.readall())
    print("Saved ->", out_path)

ResourceNotFoundError: The specified blob does not exist.
RequestId:f6fd1c21-301e-0026-4f1e-4d39c5000000
Time:2025-11-04T00:02:06.7912812Z
ErrorCode:BlobNotFound
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>BlobNotFound</Code><Message>The specified blob does not exist.
RequestId:f6fd1c21-301e-0026-4f1e-4d39c5000000
Time:2025-11-04T00:02:06.7912812Z</Message></Error>

## Explicación Notebook
Pasos concretos  
1. Ejecuta el bloque del notebook reading-db.ipynb que carga playgroundROI.gpkg y corre la consulta (SQL_QUERY) para poblar df.
2. Inspecciona columnas útiles en df: id (formato "video.mp4:person_uuid"), camera_name, timestamp, tracklet_id, total_matches.
3. Deriva video_id = df['id'].str.split(':').str[0] y agrupa por video_id, camera_name. Selecciona escenas por heurística (ej.: al menos N detecciones, o duración ≥ T segundos, o ventanas contiguas donde el gap entre frames ≤ g segundos).
4. Guardar la lista final con columnas: video_id, camera, t_start, t_end, blob_path (igual a video_id o ruta en el blob), n_detections → data/videos.csv.

### Cosas por hacer a continuación:

- Para obtener ventanas temporales más pequeñas (p. ej. ventanas de 5–10 s), en cada grupo ordena por timestamp y detecta "breaks" cuando el gap entre filas supere un umbral (p.ej. 2 s), y genera varias escenas por video/cámara.
- Prefiltra por total_matches o por el número de personas distintas (g['id_person'].nunique()) si quieres escenas con actividad humana relevante.
- Usa la columna id para extraer el blob_path exacto (si tu blob storage guarda con el mismo nombre) y luego baja/valida el blob con el bloque de Azure en reading-db.ipynb.